In [1]:
# Particle-count timing benchmark for the current Bayesian controller.
# This cell compares the existing implementation at 1k, 10k, and 100k particles.

from pathlib import Path
import csv
import importlib.util
import sys
import time

import numpy as np


# Edit only these values if a longer or shorter benchmark is needed.
PARTICLE_COUNTS = [1_000, 10_000, 100_000]
TASK_SEEDS = [7_101, 7_202, 7_303, 7_404, 7_555]
TASKS_PER_SEED = 1
UPDATE_REPEATS = 2
SELECT_REPEATS = 20
WARMUP_UPDATES = 1
WARMUP_SELECTS = 10
OUTPUT_DIR = Path.cwd() / "particle_count_timing_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def locate_benchmark_dir():
    relative = Path("review_materials") / "rl_bayesian_fixed3000_confirmatory_20260724"
    candidates = []
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        candidates.append(base / relative)
    candidates.append(Path(r"E:\GitHub\zsyEP\zsyEP\zsyEP") / relative)
    for candidate in candidates:
        if (candidate / "inputs" / "bayesian_controller.py").exists() and (candidate / "benchmark_core.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate review_materials/rl_bayesian_fixed3000_confirmatory_20260724. "
        "Run this notebook from the repository or edit the fallback path above."
    )


benchmark_dir = locate_benchmark_dir()
sys.path.insert(0, str(benchmark_dir))
from benchmark_core import StressScenario, generate_tasks


controller_path = benchmark_dir / "inputs" / "bayesian_controller.py"
spec = importlib.util.spec_from_file_location("particle_timing_bayesian_controller", controller_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Cannot import {controller_path}")
bayesian = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bayesian)


tasks = []
for seed in TASK_SEEDS:
    for task in generate_tasks(seed, TASKS_PER_SEED, StressScenario("nominal")):
        tasks.append((seed, task))


def save_state(env):
    return {
        "pKa_list": env.pKa_list.copy(),
        "pKa_std": env.pKa_std.copy(),
        "buffer_total_moles": env.buffer_total_moles.copy(),
        "buffer_total_std": env.buffer_total_std.copy(),
        "done": env.done,
        "overshoot_threshold": env.overshoot_threshold,
        "overshoot_occurred": env.overshoot_occurred,
        "overshoot_reagent": env.overshoot_reagent,
        "use_secondary_reagents": env.use_secondary_reagents,
    }


def restore_state(env, state):
    env.pKa_list[...] = state["pKa_list"]
    env.pKa_std[...] = state["pKa_std"]
    env.buffer_total_moles[...] = state["buffer_total_moles"]
    env.buffer_total_std[...] = state["buffer_total_std"]
    env.done = state["done"]
    env.overshoot_threshold = state["overshoot_threshold"]
    env.overshoot_occurred = state["overshoot_occurred"]
    env.overshoot_reagent = state["overshoot_reagent"]
    env.use_secondary_reagents = state["use_secondary_reagents"]


def prepare_case(task_seed, task, particle_count):
    # The same seed makes the initial prior state comparable across particle counts.
    np.random.seed(int(task_seed + task.task_id * 100_003) % (2**32 - 1))
    env = bayesian.PHAdjustmentEnv(num_particles=particle_count)
    env.initialize(
        task.acid_type,
        list(task.pka_values),
        task.initial_ph,
        task.target_ph,
        bayesian.MAX_STEPS,
    )
    action = env.select_best_action()[0]
    observed_ph, _, _, _ = env.step(action, mode="Simulate")
    return env, action, float(observed_ph), save_state(env)


def time_one_case(task_seed, task, particle_count):
    env, action, observed_ph, initial_state = prepare_case(task_seed, task, particle_count)

    for _ in range(WARMUP_UPDATES):
        env.update_posteriors(action, observed_ph)
        restore_state(env, initial_state)
    for _ in range(WARMUP_SELECTS):
        env.select_best_action()
        restore_state(env, initial_state)

    update_t0 = time.perf_counter_ns()
    for _ in range(UPDATE_REPEATS):
        env.update_posteriors(action, observed_ph)
        restore_state(env, initial_state)
    update_ms = (time.perf_counter_ns() - update_t0) / 1e6 / UPDATE_REPEATS

    select_t0 = time.perf_counter_ns()
    for _ in range(SELECT_REPEATS):
        env.select_best_action()
        restore_state(env, initial_state)
    select_ms = (time.perf_counter_ns() - select_t0) / 1e6 / SELECT_REPEATS

    cycle_t0 = time.perf_counter_ns()
    for _ in range(UPDATE_REPEATS):
        env.update_posteriors(action, observed_ph)
        env.select_best_action()
        restore_state(env, initial_state)
    cycle_ms = (time.perf_counter_ns() - cycle_t0) / 1e6 / UPDATE_REPEATS

    return {
        "particle_count": particle_count,
        "task_seed": task_seed,
        "task_id": task.task_id,
        "acid_type": task.acid_type,
        "initial_ph": task.initial_ph,
        "target_ph": task.target_ph,
        "update_posteriors_ms": update_ms,
        "select_best_action_ms": select_ms,
        "decision_cycle_ms": cycle_ms,
    }


rows = []
errors = []
print(f"Benchmark directory: {benchmark_dir}")
print(f"Tasks: {len(tasks)} ({TASKS_PER_SEED} per seed); particles: {PARTICLE_COUNTS}")
print("decision_cycle_ms = update_posteriors + select_best_action, using the current implementation")

for particle_count in PARTICLE_COUNTS:
    started = time.perf_counter()
    print(f"\nRunning particle_count={particle_count:,} ...", flush=True)
    for index, (task_seed, task) in enumerate(tasks, 1):
        try:
            rows.append(time_one_case(task_seed, task, particle_count))
        except Exception as exc:
            errors.append({
                "particle_count": particle_count,
                "task_seed": task_seed,
                "task_id": task.task_id,
                "error": repr(exc),
            })
            print(f"  failed task seed={task_seed}, id={task.task_id}: {exc}", flush=True)
        if index == 1 or index % 5 == 0 or index == len(tasks):
            print(f"  {index}/{len(tasks)}", flush=True)
    print(f"  elapsed: {time.perf_counter() - started:.1f} s", flush=True)


raw_path = OUTPUT_DIR / "particle_count_timing_per_task.csv"
summary_path = OUTPUT_DIR / "particle_count_timing_summary.csv"
error_path = OUTPUT_DIR / "particle_count_timing_errors.csv"

if rows:
    with raw_path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

summary_rows = []
for particle_count in PARTICLE_COUNTS:
    subset = [row for row in rows if row["particle_count"] == particle_count]
    if not subset:
        continue
    item = {"particle_count": particle_count, "cases": len(subset)}
    for metric in ("update_posteriors_ms", "select_best_action_ms", "decision_cycle_ms"):
        values = np.asarray([row[metric] for row in subset], dtype=float)
        item[f"{metric}_mean"] = float(values.mean())
        item[f"{metric}_median"] = float(np.median(values))
        item[f"{metric}_sd"] = float(values.std(ddof=1)) if len(values) > 1 else 0.0
        item[f"{metric}_p95"] = float(np.percentile(values, 95))
    summary_rows.append(item)

if summary_rows:
    with summary_path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(summary_rows[0]))
        writer.writeheader()
        writer.writerows(summary_rows)
if errors:
    with error_path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(errors[0]))
        writer.writeheader()
        writer.writerows(errors)


print("\nSummary (milliseconds per decision cycle):")
for item in summary_rows:
    print(
        f"  {item['particle_count']:>7,} particles | "
        f"update median {item['update_posteriors_ms_median']:.3f} | "
        f"select median {item['select_best_action_ms_median']:.3f} | "
        f"cycle median {item['decision_cycle_ms_median']:.3f} | "
        f"cycle p95 {item['decision_cycle_ms_p95']:.3f}"
    )

if summary_rows:
    baseline = summary_rows[0]["decision_cycle_ms_median"]
    print("\nScaling relative to the first particle count:")
    for item in summary_rows:
        ratio = item["decision_cycle_ms_median"] / baseline if baseline > 0 else float("nan")
        print(f"  {item['particle_count']:>7,} particles: {ratio:.2f}x")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    x = np.asarray([item["particle_count"] for item in summary_rows], dtype=float)
    plt.figure(figsize=(7.2, 4.6))
    for metric, label in (
        ("update_posteriors_ms_median", "posterior update"),
        ("decision_cycle_ms_median", "decision cycle"),
    ):
        plt.loglog(x, [item[metric] for item in summary_rows], marker="o", label=label)
    plt.xlabel("Number of particles")
    plt.ylabel("Median time (ms)")
    plt.title("Bayesian controller timing versus particle count")
    plt.grid(True, which="both", alpha=0.25)
    plt.legend()
    plt.tight_layout()
    figure_path = OUTPUT_DIR / "particle_count_timing_scaling.png"
    plt.savefig(figure_path, dpi=200)
    plt.close()
    print(f"Figure: {figure_path}")
except Exception as exc:
    print(f"Plot skipped: {exc}")

print(f"Per-task CSV: {raw_path}")
print(f"Summary CSV: {summary_path}")
if errors:
    print(f"Errors CSV: {error_path}")


Benchmark directory: C:\Users\ZSY\Desktop\FDTD\particle_count_timing_processed_20260806\review_materials\rl_bayesian_fixed3000_confirmatory_20260724
Tasks: 5 (1 per seed); particles: [1000, 10000, 100000]
decision_cycle_ms = update_posteriors + select_best_action, using the current implementation

Running particle_count=1,000 ...


  1/5


  5/5


  elapsed: 2.5 s



Running particle_count=10,000 ...


...

  1/5


  5/5


  elapsed: 51.7 s



Summary (milliseconds per decision cycle):
    1,000 particles | update median 19.473 | select median 2.056 | cycle median 26.188 | cycle p95 69.481
   10,000 particles | update median 167.235 | select median 1.651 | cycle median 194.206 | cycle p95 204.728
  100,000 particles | update median 1935.485 | select median 1.986 | cycle median 2016.150 | cycle p95 2182.639

Scaling relative to the first particle count:
    1,000 particles: 1.00x
   10,000 particles: 7.42x
  100,000 particles: 76.99x


Figure: C:\Users\ZSY\Desktop\FDTD\particle_count_timing_processed_20260806\particle_count_timing_results\particle_count_timing_scaling.png
Per-task CSV: C:\Users\ZSY\Desktop\FDTD\particle_count_timing_processed_20260806\particle_count_timing_results\particle_count_timing_per_task.csv
Summary CSV: C:\Users\ZSY\Desktop\FDTD\particle_count_timing_processed_20260806\particle_count_timing_results\particle_count_timing_summary.csv
